In [ ]:
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt

from static_training import train_static
from progressive_training import train_progressive
from plot3HeatMaps import plot_three_heatmaps
from mistake_driven_training import train_mistake_driven

In [ ]:
# Load a single file first (or subset)
df = pd.read_csv("data/airlines/2010.csv")

df.head()

In [ ]:
# Drop useless column
df = df.drop(columns=['Unnamed: 27'], errors='ignore')

# Remove cancelled/diverted flights
df = df[df['CANCELLED'] == 0]
df = df[df['DIVERTED'] == 0]

# Remove rows that don't have an arrival delay value (Still keeps zero)
df = df[df['ARR_DELAY'].notna()]

In [ ]:
drop_cols = [
    'ACTUAL_ELAPSED_TIME',
    'ARR_DELAY',       # target
    'ARR_TIME',
    'TAXI_IN',
    'WHEELS_ON'
]

y = df['ARR_DELAY']
X = df.drop(columns=drop_cols + ['FL_DATE'])

One HotEncode Carrier, Destination, Origin

In [ ]:
# Encode OP_CARRIER
X = pd.get_dummies(X, columns=['OP_CARRIER'], drop_first=False)

# Label encode ORIGIN and DEST (avoid huge one-hot)
from sklearn.preprocessing import LabelEncoder

le_origin = LabelEncoder()
le_dest = LabelEncoder()

X['ORIGIN'] = le_origin.fit_transform(X['ORIGIN'])
X['DEST'] = le_dest.fit_transform(X['DEST'])

In [ ]:
df = df.sort_values('FL_DATE')

X = X.loc[df.index]
y = y.loc[df.index]

In [ ]:
look_back_values = [5000, 10000, 20000] 
tree_depth = 5

look_ahead = [5000, 10000, 20000]
retrain_step = 5000
threshold = 20          

# Static Model

In [ ]:
results_static = {}
time_static = {}

for lb in look_back_values:
    results_static[lb] = {}
    time_static[lb] = {}
    
    for la in look_ahead:
        start_time = time.time()
        
        _, avg_rmse = train_static(X, y, look_back=lb,look_ahead= la,  depthOfTree=tree_depth)
        
        elapsed_time = time.time() - start_time
        
        results_static[lb][la] = avg_rmse
        time_static[lb][la] = elapsed_time
        
        print(f"Look-Back {lb}, Look-Ahead {la} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

# Progressive Model

In [ ]:
results_progressive = {}
time_progressive = {}

for lb in look_back_values:
    results_progressive[lb] = {}
    time_progressive[lb] = {}
    
    for la in look_ahead:
        start_time = time.time()
        
        _, avg_rmse = train_progressive(
            X, y,
            look_back=lb,
            look_ahead=la,
            retrain_step=retrain_step,
            depthOfTree=tree_depth
        )
        
        elapsed_time = time.time() - start_time
        
        results_progressive[lb][la] = avg_rmse
        time_progressive[lb][la] = elapsed_time
        
        print(f"Look-Back {lb}, Look-Ahead {la} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

#  Mistake Model

In [ ]:
results_mistake = {}
time_mistake = {}

for lb in look_back_values:
    results_mistake[lb] = {}
    time_mistake[lb] = {}
    
    for la in look_ahead:
        start_time = time.time()
        
        _, avg_rmse = train_mistake_driven(
            X, y,
            look_back=lb,
            look_ahead=la,
            depthOfTree=tree_depth,
            threshold=threshold
        )
        
        elapsed_time = time.time() - start_time
        
        results_mistake[lb][la] = avg_rmse
        time_mistake[lb][la] = elapsed_time
        
        print(f"Look-back {lb}, Look-Ahead {la} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

In [ ]:
df_static_rmse = pd.DataFrame(results_static).T
df_progressive_rmse = pd.DataFrame(results_progressive).T
df_mistake_rmse = pd.DataFrame(results_mistake).T

df_static_time = pd.DataFrame(time_static).T
df_progressive_time = pd.DataFrame(time_progressive).T
df_mistake_time = pd.DataFrame(time_mistake).T


In [ ]:
plot_three_heatmaps(
    dfs=[df_static_rmse, df_progressive_rmse, df_mistake_rmse],
    titles=["Static RMSE", "Progressive RMSE", "Mistake-Driven RMSE"],
    vmin=min(df_static_rmse.min().min(), df_progressive_rmse.min().min(), df_mistake_rmse.min().min()),
    vmax=max(df_static_rmse.max().max(), df_progressive_rmse.max().max(), df_mistake_rmse.max().max())
)


plot_three_heatmaps(
    dfs=[df_static_time, df_progressive_time, df_mistake_time],
    titles=["Static Time (s)", "Progressive Time (s)", "Mistake-Driven Time (s)"],
    vmin=min(df_static_time.min().min(), df_progressive_time.min().min(), df_mistake_time.min().min()),
    vmax=max(df_static_time.max().max(), df_progressive_time.max().max(), df_mistake_time.max().max())
)

# Conclusion
Mistake driven training is able to maintain a competitive edge with progressive training while providing a substantial performance boost.